# 03 · GenAI Basics — your first calls to the Gemini API

**Workshop:** AI for Actuaries — From Foundations to AI Agents  
**Session / Part:** S2.P1  
**Slides covered:** S2.P1.10  
**Author:** Dr Rohan Yashraj Gupta (FIA, FIAI), with Satya Sai Mudigonda and Sai Krishna Vadali  
**Workshop date:** 15 May 2026 · Hilton near Airport, Mumbai  
**License:** CC BY-NC 4.0 — for educational use within the IFoA workshop and follow-up case study

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rohanyashraj/ifoa-workshop/blob/main/notebooks/03_genai_basics.ipynb)

## What this notebook does
Three things, in order:

1. **First call.** A single Gemini text-generation call from a Colab runtime. You'll see how the SDK auth pattern works with Colab Secrets.
2. **Structured JSON.** The same model, but constrained to return a small JSON object describing one ABC Motor risk factor — the pattern you'll use whenever you want machine-readable output rather than prose.
3. **A live hallucination demo.** We deliberately ask Gemini for a regulation that does not exist (an "IRDAI motor tariff factor for hatchbacks under 1000 cc, 2024") and watch the model invent one. This is the demo previewed on slide S2.P1.7.

## Prerequisites
- A Google account (for Colab).
- A Gemini API key — instructions are in the **API setup** cell below. The free tier is sufficient for everything in this notebook.
- No local install required.

## How to run
Top menu → **Runtime → Run all**. The first cell installs dependencies; subsequent cells should run without intervention.

## ⚠️ Pre-workshop verification — read before workshop day

The `google-genai` version pin and the `gemini-2.5-flash` model identifier in this notebook are **placeholders**. Both must be re-verified against a fresh Colab runtime before the workshop:

1. **`google-genai` version.** Run a clean `!pip install -q google-genai` (no pin), then `!pip show google-genai` to capture the version that resolves cleanly. Update the pin in the install cell below.
2. **Model identifier.** Per `00_citations.md` entry **D2**, confirm the exact Gemini model string in production at workshop time — the `gemini-2.5-flash` placeholder may need to be updated.

Once both are confirmed, delete this cell.

In [2]:
# Install the Gemini Python SDK.
# Pinned version is a placeholder — verify on a clean Colab runtime before workshop day (see cell above).
!pip install -q google-genai

In [3]:
# === Standard imports ===
import os
import json
import numpy as np

# Reproducibility — note: LLM calls are NOT deterministic even with this seed (slide S2.P1.6).
SEED = 42
np.random.seed(SEED)

## 1 · API setup

We read the Gemini API key from **Colab Secrets** rather than pasting it into the notebook. The key icon in the left sidebar of Colab opens the Secrets panel; add a secret named `GOOGLE_API_KEY` with your key as the value, and toggle "Notebook access" on.

If you're running this notebook outside Colab, set the `GOOGLE_API_KEY` environment variable instead.

In [9]:
from dotenv import load_dotenv
import os

def load_api_key():
    api_key = None

    # Colab: Try google.colab.userdata (interactive secret storage)
    try:
        from google.colab import userdata
        potential_api_key = userdata.get('GEMINI_API_KEY') # Changed to GOOGLE_API_KEY
        if potential_api_key:
            api_key = potential_api_key
            os.environ['GOOGLE_API_KEY'] = api_key  # Changed to GOOGLE_API_KEY
            print("Loaded API key from Colab Secrets.")
            return
    except (ImportError, Exception):
        pass  # Not in Colab or Colab Secrets not accessible, continue to local fallback

    # Local: Try environment variable directly or from a .env file
    if not api_key:
        load_dotenv()  # Loads variables from .env into environment
        potential_api_key = os.getenv('GOOGLE_API_KEY') # Changed to GOOGLE_API_KEY
        if potential_api_key:
            api_key = potential_api_key
            os.environ['GOOGLE_API_KEY'] = api_key  # Changed to GOOGLE_API_KEY
            print("Loaded API key from environment variable.")
            return

    # If API key is still not found, raise an error
    raise RuntimeError(
        'GOOGLE_API_KEY is not set. ' # Changed to GOOGLE_API_KEY
        'Add it via Colab Secrets, set the env variable, or add it to a .env file.'
    )

load_api_key()

Loaded API key from Colab Secrets.


## 2 · First call — text generation

We instantiate a `genai.Client` (the SDK reads `GOOGLE_API_KEY` from the environment automatically), then call `client.models.generate_content` with a model identifier and a prompt. The model returns a response object; the generated text is on `response.text`.

The model string `gemini-2.5-flash` is a placeholder — see the warning at the top of the notebook.

In [11]:
from google import genai

# The SDK picks up GOOGLE_API_KEY from the environment automatically.
client = genai.Client()

# Pin the model string — see citation D2 (00_citations.md).
MODEL_ID = 'gemini-2.5-flash'

response = client.models.generate_content(
    model=MODEL_ID,
    contents='Define IBNR for a non-actuarial board member, in one line.',
)

print(response.text)

IBNR is the estimated financial liability for claims that have occurred but have not yet been reported to the company.


## 3 · Structured JSON output

Prose is fine for humans, but the moment you want a downstream pipeline to consume the output, you want structure. The Gemini SDK supports a JSON response mode where you describe the schema you want — either with a Pydantic model or by passing a JSON Schema dictionary — and the model is constrained to return data matching that shape.

We use the dictionary form here so this notebook has zero non-essential dependencies. The example asks Gemini to describe **vehicle age** as a risk factor in the ABC Motor 2024 book — using terminology consistent with `00_personas_and_datasets.md`.

In [ ]:
# A small schema describing one motor risk factor.
factor_schema = {
    'type': 'object',
    'properties': {
        'name':                 {'type': 'string'},
        'short_description':    {'type': 'string'},
        'typical_levels':       {'type': 'array', 'items': {'type': 'string'}},
        'expected_direction':   {'type': 'string',
                                 'enum': ['increases_frequency',
                                          'decreases_frequency',
                                          'mixed_or_non_monotonic']},
        'actuarial_caveat':     {'type': 'string'},
    },
    'required': ['name', 'short_description', 'typical_levels',
                 'expected_direction', 'actuarial_caveat'],
}

prompt = (
    'You are an actuarial assistant. Describe vehicle age as a frequency '
    'risk factor for a private car comprehensive book in India. '
    'Return only the requested JSON fields — no commentary.'
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config={
        'response_mime_type': 'application/json',
        'response_schema': factor_schema,
    },
)

factor = json.loads(response.text)
print(json.dumps(factor, indent=2))

## 4 · Hallucination demo (deliberate)

**This cell is intentionally producing nonsense.** It exists so you see — once, with your own eyes — what a confident hallucination from a frontier model looks like.

We ask Gemini for the *"IRDAI motor tariff factor for hatchbacks under cubic capacity 1000 in 2024"*. There is no such published, single, authoritative number. India's general insurance motor own-damage market has been **detariffed since 2007**; current pricing is filed product by product under IRDAI's File-and-Use / Use-and-File frameworks, not as a tariff factor by body type. The honest answer is "that question rests on a false premise". A well-behaved model would say so. A typical LLM, asked confidently, will produce a confident-sounding number.

Run the cell. Read what you get. Then read the explanation below it.

In [ ]:
hallucination_prompt = (
    'Quote the exact IRDAI motor tariff loading factor for hatchback '
    'private cars with cubic capacity below 1000 cc, as published for '
    'the 2024 calendar year. Give the numeric factor, the section reference '
    'in the IRDAI document, and the date the circular was issued.'
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=hallucination_prompt,
)

print('=== MODEL OUTPUT (likely contains hallucinated specifics) ===')
print(response.text)
print('=== END ===')

### Why this is dangerous, and what to do about it

Three things to take from the cell above:

**1. The model didn't refuse — it invented.** It produced a numeric factor, a section reference, and a circular date, with full fluency. None of those are real. India detariffed motor own-damage in 2007, and there is no IRDAI "loading factor by body type and CC band" published for 2024. The model is not lying in any deliberate sense — it's doing exactly what an LLM does, which is sample plausible next tokens. Plausible is not the same as true.

**2. Re-running the cell will give you a different fabrication.** That's slide S2.P1.6 in action — same prompt, different output. If you ran this cell a hundred times you'd get a hundred different fake factors, dates, and circular references, all equally confident.

**3. The fix is procedural, not technical.**

- **Ground every factual claim.** If you need a regulation, retrieve it. Hand the LLM the actual IRDAI circular (via RAG, file upload, or a search tool) and ask it to summarise — never ask it to recall.
- **Verify before it ships.** Any number that comes out of an LLM and ends up in a board pack or a tariff filing must trace to a primary source you can name and link.
- **Log the prompt, the model, the timestamp, and the response.** When (not if) something is queried later, you need to be able to reproduce the chain.

The actuary still owns the number. That principle does not move because the tool got smarter.

In [ ]:
print('03_genai_basics.ipynb — first Gemini call, structured JSON, and a live hallucination, all in fewer than 50 lines of Python.')

## Wrap-up

You should now be able to:

- Authenticate to the Gemini API from a Colab notebook using Colab Secrets, with a clean local fallback.
- Make a basic text-generation call and read `response.text`.
- Constrain the model to return structured JSON matching a schema you define.
- Recognise a hallucinated specific — a number, a citation, a date — and explain why it appeared.

**Where to next:** open `04_genai_tools.ipynb` for three reproducible Gemini prompts that scaffold actuarial documentation tasks (Demo 1: ABC Motor risk-factor table; Demo 2: ABC Term Life underwriting guidelines; Demo 3: ABC Health board-pack summary).

**Companion slides:** S2.P1.10 of the deck.

**Questions?** Bring them to the live Q&A or post in the workshop Slack channel.